# ch06 练习：聚类与无监督

与讲解版逐 Cell 对应。核心代码被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即自查通过。卡住就回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 无监督建模 | KMeans / 层次 / DBSCAN / GMM 的 fit-predict |
| 聚类评估 | inertia / 轮廓系数 / ARI |

## 二、API 速查

| 类 | 关键参数 | 一句话 |
|---|---|---|
| `KMeans(n_clusters, n_init)` | `random_state` | `predict` / `transform` 可服务化 |
| `AgglomerativeClustering(linkage)` | — | 无 `predict` |
| `DBSCAN(eps, min_samples)` | — | 噪声=-1，eps 敏感 |
| `GaussianMixture(n_components)` | — | 软分配 `predict_proba` |
| `silhouette_score` / `silhouette_samples` | — | 整体 / 样本级轮廓 |
| `adjusted_rand_score` | — | 编号置换免疫的对账尺 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

DATA = Path("data")

df = pd.read_csv(DATA / "station_load.csv")   # 300 台区 × 5 特征 + 用电类型（仅用于对账）

FEATS = ["日均负荷", "夜间负荷率", "峰谷差率", "周末负荷率", "报装容量kVA"]
TRUE = "用电类型"
X = df[FEATS]
y_true = df[TRUE]

ss = StandardScaler().fit(X)
Xs = ss.transform(X)

print("sklearn", sklearn.__version__)
print(df.shape, "| 类型计数:", y_true.value_counts().to_dict())
print("量纲对比: 日均负荷 std", round(float(X["日均负荷"].std()), 1),
      "vs 峰谷差率 std", round(float(X["峰谷差率"].std()), 4))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def cluster_table(y_true, labels):
    """交叉表 + 最优映射准确率（簇编号与真实标签的对应关系任选）。"""
    ct = pd.crosstab(y_true, labels)
    mapping = ct.idxmax(axis=1).to_dict()          # 真实类型 -> 簇编号
    acc = sum(ct.loc[t, c] for t, c in mapping.items()) / len(y_true)
    return ct, {c: t for t, c in mapping.items()}, acc


print("脚手架就绪：probe / cluster_table")

## 题 1：KMeans 完美聚类与新样本归属（对应讲解 §3.1）

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(Xs)
ct, clu2type, acc = cluster_table(y_true, km.labels_)
print("inertia(WCSS):", round(float(km.inertia_), 4),
      "| n_iter:", int(km.n_iter_))
print("簇计数:", pd.Series(km.labels_).value_counts().sort_index().to_dict())
print(ct.to_string())
print("ARI vs 用电类型:", round(adjusted_rand_score(y_true, km.labels_), 4),
      "| 最优映射 acc:", round(acc, 4))
print("簇 -> 类型:", clu2type)

# 商业画像新样本（脚手架，不用挖空）
new = pd.DataFrame({"日均负荷": [510.0], "夜间负荷率": [0.22], "峰谷差率": [0.74],
                    "周末负荷率": [0.66], "报装容量kVA": [800.0]})

new_s = ss.transform(new)
lab_new = int(km.predict(new_s)[0])
print("新样本(商业画像) → 簇", lab_new, "=", clu2type[lab_new],
      "| 到各簇心距离:", np.round(km.transform(new_s)[0], 4).tolist())

assert round(float(km.inertia_), 4) == 402.6153
assert pd.Series(km.labels_).value_counts().sort_index().to_dict() == {0: 90, 1: 75, 2: 135}
assert round(adjusted_rand_score(y_true, km.labels_), 4) == 1.0, "ARI=1 完美还原三型"
assert round(acc, 4) == 1.0
assert clu2type == {0: "商业型", 1: "工业型", 2: "居民型"}
assert lab_new == 0 and clu2type[lab_new] == "商业型"

**为什么这样做**：ARI 不受簇编号置换影响，交叉表 + 最优映射给出「哪个簇
是哪类人」的业务语言——两把尺子一起拿，汇报才完整。

## 题 2：量纲陷阱（对应讲解 §3.2）

In [ ]:
km_raw = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X.to_numpy())
ct_raw, _, acc_raw = cluster_table(y_true, km_raw.labels_)
print("raw inertia:", round(float(km_raw.inertia_), 2),
      "| scaled inertia:", round(float(km.inertia_), 4))
print("raw ARI:", round(adjusted_rand_score(y_true, km_raw.labels_), 4),
      "| raw 最优映射 acc:", round(acc_raw, 4))
print(ct_raw.to_string())
print("解读: 报装容量(数百) 与 率值(零点几) 量纲差 3 个数量级，距离被容量独裁")

assert round(adjusted_rand_score(y_true, km_raw.labels_), 4) == 0.2557
assert round(acc_raw, 4) == 0.61, "近四成台区被分错型，商业/居民还在抢同一个簇"
assert round(float(km_raw.inertia_), 2) == 23988840.36, "inertia 是量纲的奴隶，跨尺度不可比"

## 题 3：k 扫描——肘部与轮廓（对应讲解 §3.3）

In [ ]:
inertias, sils, ks = [], [], list(range(2, 9))

# 扫描循环（骨架不用挖空）
for k in ks:
    kmk = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    inertias.append(round(float(kmk.inertia_), 2))
    sils.append(round(float(silhouette_score(Xs, kmk.labels_)), 4))
    print(f"k={k}: inertia={inertias[-1]:>9.2f} silhouette={sils[-1]:.4f}")

best_k = ks[int(np.argmax(sils))]
print("轮廓系数峰值 k =", best_k, "→", sils[int(np.argmax(sils))])

assert inertias == [890.25, 402.62, 293.56, 218.98, 172.42, 155.54, 141.55], "单调下降没有拐点喊得那么响"
assert sils == [0.4291, 0.5628, 0.5156, 0.4956, 0.4592, 0.3907, 0.3523]
assert best_k == 3, "轮廓系数在 k=3 出峰，与真实簇数一致"

## 题 4：样本级轮廓（对应讲解 §3.4）

In [ ]:
sil_vals = silhouette_samples(Xs, km.labels_)
per = pd.Series(sil_vals).groupby(km.labels_).mean().round(4)
print("各簇平均轮廓:", per.to_dict())
print("overall:", round(float(silhouette_score(Xs, km.labels_)), 4))
print("最低样本轮廓:", round(float(sil_vals.min()), 4),
      "| 负轮廓样本数:", int((sil_vals < 0).sum()))

assert per.to_dict() == {0: 0.5381, 1: 0.5907, 2: 0.5638}
assert round(float(silhouette_score(Xs, km.labels_)), 4) == 0.5628
assert int((sil_vals < 0).sum()) == 0, "没有样本分错阵营"

## 题 5：层次聚类与编号抽签（对应讲解 §3.5）

In [ ]:
# 循环骨架不用挖空；挖空的是循环体
for link in ("ward", "complete", "average"):
    ag = AgglomerativeClustering(n_clusters=3, linkage=link).fit(Xs)
    same = float((ag.labels_ == km.labels_).mean())
    print(f"{link:>9}: ARI={adjusted_rand_score(y_true, ag.labels_):.4f} "
          f"| 与 KMeans 标签逐位一致率: {same:.4f}")

ag_ward = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(Xs)
st_pred = probe(getattr, AgglomerativeClustering(), "predict")
print("AGG.predict →", st_pred, "← 层次聚类没有 predict，新样本要另想办法")

assert round(adjusted_rand_score(y_true, ag_ward.labels_), 4) == 1.0
assert float((ag_ward.labels_ == km.labels_).mean()) == 0.0, "ARI=1.0 但编号完全错位——簇编号无语义"
assert st_pred == ("err", "AttributeError")

## 题 6：DBSCAN 的 eps（对应讲解 §3.6）

In [ ]:
db = DBSCAN(eps=0.3, min_samples=5).fit(Xs)
n03 = int((db.labels_ == -1).sum())
print(f"eps=0.3: 噪声 {n03}/300，只剩 {len(set(db.labels_)) - 1} 个簇 ← 全员社死")

# eps 扫描（不用挖空）：噪声海 → 碎片化 → 基本成型 → 完美
results = {}
for eps in (0.5, 0.8, 1.2):
    dbp = DBSCAN(eps=eps, min_samples=5).fit(Xs)
    lab = dbp.labels_
    n_cl = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = int((lab == -1).sum())
    ari = round(float(adjusted_rand_score(y_true[lab != -1], lab[lab != -1])), 4) if n_cl > 1 else 0.0
    results[eps] = (n_cl, n_noise, ari)
    print(f"eps={eps}: 簇数={n_cl} 噪声={n_noise} ARI(非噪声)={ari}")

assert n03 == 289, "eps 太小：289 个点被判噪声"
assert results[0.5] == (9, 111, 0.3366), "eps 稍大：碎成 9 瓣"
assert results[0.8] == (5, 4, 0.9081)
assert results[1.2] == (3, 0, 1.0), "eps=1.2 才完整还原三簇"

## 题 7：k-距离图选 eps（对应讲解 §3.7）

In [ ]:
nn = NearestNeighbors(n_neighbors=5).fit(Xs)
d4 = np.sort(nn.kneighbors(Xs)[0][:, -1])          # 每个点的第 4 近邻距离
p50, p90, p95 = (round(float(np.quantile(d4, q)), 4) for q in (0.5, 0.9, 0.95))
print(f"第4近邻距离分位: p50={p50} p90={p90} p95={p95}")
print("经验法则: min_samples=5 时，eps 取曲线拐点附近 ≈ 1.0~1.2")

assert p50 == 0.5348 and p90 == 0.7546 and p95 == 0.8173
assert p95 < 1.2, "拐点之下才解释了 eps=1.2 为什么不坍缩"

## 题 8：GMM 软分配（对应讲解 §3.8）

In [ ]:
gm = GaussianMixture(n_components=3, random_state=42).fit(Xs)
lab_g = gm.predict(Xs)
proba_g = gm.predict_proba(Xs)
print("GMM ARI:", round(adjusted_rand_score(y_true, lab_g), 4),
      "| 计数:", pd.Series(lab_g).value_counts().sort_index().to_dict())
print("proba shape:", proba_g.shape, "| 行和:", np.round(proba_g.sum(axis=1)[:3], 6).tolist())
print("新样本(商业画像) GMM 概率:", np.round(gm.predict_proba(new_s)[0], 4).tolist(),
      "→", ["商业型", "工业型", "居民型"][int(np.argmax(gm.predict_proba(new_s)[0]))])

assert round(adjusted_rand_score(y_true, lab_g), 4) == 1.0
assert proba_g.shape == (300, 3)
assert float(np.abs(proba_g.sum(axis=1) - 1).max()) < 1e-6
assert int(np.argmax(gm.predict_proba(new_s)[0])) == 0, "商业画像落在商业簇"

## 收官自查

1. 题 1 的 ARI=1.0 和 acc=1.0 分别在说什么？去掉「用电类型」列还剩哪个能算？
2. 题 2 里 raw 版也报出了 3 个簇，为什么说它「失忆」？
3. 题 3 的轮廓系数 0.5628 离 1 很远——这个聚类到底好不好？
4. 题 5 的 ward 与 KMeans 一致率 0.0，谁对谁错？
5. 题 8 的 GMM 概率 0.92 意味着什么？什么业务场景必须用它而不是 KMeans？

全部答得上来，进入 ch07（分类评估）。